# 3 · Analysis & Figures

Load the BB84 results from the FABRIC run, visualize the photon pipeline, compare against
theory, cross-validate against QFabric's simulation, and break down the photon budget.

**Prerequisite:** run `02_run_experiment` on a slice first — it writes
`results/fabric_*_results.json`, which this notebook reads. Run outputs are not tracked
in the repo, so there is nothing to load until you have made a measurement.

### At a glance
- **Purpose:** turn the recorded results into the figures and tables (photon pipeline, theory comparison, loss budget, cross-platform bars).
- **Inputs:** `results/fabric_*_results.json` from notebook fabric/02 and (optional) `results/all_scenarios.json` from notebook fabric/04.
- **Outputs:** inline plots and summary tables.
- **Runs on / runtime:** anywhere, but **only after notebook fabric/02 has produced results on a slice**; < 1 min.
- **If something fails:** a missing `fabric_*_results.json` means notebook fabric/02 has not been run — the load cell below says so and stops. If `all_scenarios.json` is absent (or has no row for this scenario), Section 5 falls back to FABRIC-vs-local-sim and says so.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PROJECT_DIR = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'qne').is_dir())
sys.path.insert(0, str(PROJECT_DIR))

from qne.metrics import ExperimentMetrics
from qne.config import ScenarioConfig
from validation.scenario import ValidationScenario
from validation.run_qfabric import run_qfabric_bb84_simulated

%matplotlib inline
plt.rcParams.update({"figure.dpi": 120, "font.size": 11})

## 1. Load FABRIC Results

In [ ]:
alice_path = PROJECT_DIR / "results" / "fabric_alice_results.json"
bob_path = PROJECT_DIR / "results" / "fabric_bob_results.json"
missing = [p for p in (alice_path, bob_path) if not p.exists()]
if missing:
    raise FileNotFoundError(
        "No run results found: " + ", ".join(p.name for p in missing) + ".\n"
        "Run outputs are not tracked in the repo. Execute notebook "
        "`fabric/02_run_experiment` on a slice first (or point PROJECT_DIR at a "
        "checkout that has them)."
    )

alice = ExperimentMetrics.from_json(alice_path)
bob = ExperimentMetrics.from_json(bob_path)

# Alice has the authoritative sent count; Bob has the receiver perspective
metrics = alice
print(f"Loaded scenario: {metrics.scenario_name}")
print(f"Photons sent: {metrics.photons_sent:,}")
print(f"Photons received: {metrics.photons_received:,}")
print(f"Sifted bits: {metrics.sifted_bits:,}")
print(f"Final key bits: {metrics.final_key_bits:,}")
print(f"QBER: {metrics.qber:.4f}")
print(f"Elapsed: {metrics.elapsed_seconds:.1f}s")

## 2. Experiment Summary

In [ ]:
cfg = metrics.config
summary = pd.DataFrame([
    ["Scenario", metrics.scenario_name],
    ["Slice", metrics.config.get("slice", "single-site (distance emulated)")],
    ["Distance (km)", cfg["channel"]["distance_km"]],
    ["Attenuation (dB/km)", cfg["channel"]["attenuation_db_per_km"]],
    ["Detector efficiency", cfg["detector"]["efficiency"]],
    ["Dark count rate (Hz)", cfg["detector"]["dark_count_rate"]],
    ["Photons sent", f"{metrics.photons_sent:,}"],
    ["Sample fraction", cfg["protocol"]["sample_fraction"]],
    ["Seed", cfg.get("seed", "N/A")],
    ["Start time", metrics.start_time],
    ["End time", metrics.end_time],
    ["Elapsed (s)", f"{metrics.elapsed_seconds:.1f}"],
], columns=["Parameter", "Value"])

summary.style.hide(axis="index")

## 3. Photon Pipeline Waterfall

Track photon counts through each stage of the BB84 pipeline, with
theoretical expectations overlaid.

In [ ]:
# Reconstruct the pipeline from config
sc = ScenarioConfig.from_dict(cfg)
n_sent = metrics.photons_sent
channel_loss_prob = sc.loss_probability
det_eff = sc.detector.efficiency

# Theoretical expectations
theory_after_channel = n_sent * (1 - channel_loss_prob)
theory_detected = theory_after_channel * det_eff
theory_sifted = theory_detected * 0.5  # basis match probability
theory_final = theory_sifted * (1 - cfg["protocol"]["sample_fraction"])

# Actual counts
actual_after_channel = metrics.photons_received / det_eff  # back out detector effect
actual_detected = metrics.photons_received
actual_sifted = metrics.sifted_bits
actual_final = metrics.final_key_bits

stages = ["Sent", "Channel\nsurvivors", "Detected", "Sifted", "Final key"]
actual_vals = [n_sent, actual_after_channel, actual_detected, actual_sifted, actual_final]
theory_vals = [n_sent, theory_after_channel, theory_detected, theory_sifted, theory_final]

x = np.arange(len(stages))
fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(x, actual_vals, width=0.5, color="steelblue", alpha=0.85, label="FABRIC actual")
ax.plot(x, theory_vals, "o--", color="darkorange", linewidth=2, markersize=8, label="Theory")

for i, (a, t) in enumerate(zip(actual_vals, theory_vals)):
    ax.text(i, a + n_sent * 0.02, f"{a:,.0f}", ha="center", va="bottom", fontsize=9, color="steelblue")
    ax.text(i, t - n_sent * 0.05, f"{t:,.0f}", ha="center", va="top", fontsize=9, color="darkorange")

ax.set_xticks(x)
ax.set_xticklabels(stages)
ax.set_ylabel("Photon / bit count")
ax.set_title("BB84 Photon Pipeline: FABRIC vs Theory")
ax.legend()
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()

## 4. Theory Comparison Table

Compare actual FABRIC results against theoretical predictions derived from
the scenario configuration.

In [ ]:
# Compute actual rates
actual_channel_loss = metrics.photons_lost / metrics.photons_sent
# Combined loss = channel + detector; separate detector contribution
actual_detection_rate = metrics.photons_received / metrics.photons_sent
actual_basis_match = metrics.sifted_bits / metrics.photons_received if metrics.photons_received else 0

# Expected values
expected_channel_loss = channel_loss_prob
expected_overall_detection = (1 - channel_loss_prob) * det_eff
expected_basis_match = 0.5

comparison = pd.DataFrame([
    ["Channel loss",
     f"{expected_channel_loss:.4f} ({expected_channel_loss*100:.2f}%)",
     f"{actual_channel_loss:.4f} ({actual_channel_loss*100:.2f}%)",
     f"{abs(actual_channel_loss - expected_channel_loss):.4f}"],
    ["Overall detection rate",
     f"{expected_overall_detection:.4f} ({expected_overall_detection*100:.2f}%)",
     f"{actual_detection_rate:.4f} ({actual_detection_rate*100:.2f}%)",
     f"{abs(actual_detection_rate - expected_overall_detection):.4f}"],
    ["Basis match rate",
     f"{expected_basis_match:.4f} ({expected_basis_match*100:.1f}%)",
     f"{actual_basis_match:.4f} ({actual_basis_match*100:.2f}%)",
     f"{abs(actual_basis_match - expected_basis_match):.4f}"],
    ["QBER",
     f"{(1 - cfg['channel']['polarization_fidelity'])/2:.4f} (intrinsic, (1-F)/2)",
     f"{metrics.qber:.4f} ({metrics.qber*100:.2f}%)",
     f"{metrics.qber:.4f}"],
    ["Secure key rate (bits/photon)",
     "-",
     f"{metrics.secure_key_rate:.4f}",
     "-"],
], columns=["Metric", "Expected", "Actual (FABRIC)", "|Delta|"])

comparison.style.hide(axis="index")

## 5. Cross-Platform Comparison

Compare QBER and secure key rate across **all backends** from the on-FABRIC
cross-validation (notebook fabric/04): QFabric measured, QFabric-sim, SeQUeNCe, NetSquid.
Backends come from `results/all_scenarios.json`, which notebook fabric/04 writes on the
slice. If that file is absent — or has no row for this scenario, or every backend in it
errored — this falls back to FABRIC measured vs a local QFabric simulation and says so.

In [ ]:
import json

# Notebook fabric/04 records the on-slice cross-validation to all_scenarios.json
# (there is no cross_validation.json — nothing writes one). Take the row for this
# run's scenario, but only if it describes the same physics and carries real data;
# otherwise fall back to a local sim comparison and say why.

def _usable(b):
    """The same standard notebook fabric/04 applies. A recorded error, or a run
    that sifted nothing, is not data — a backend reporting qber=0.0 off zero
    sifted bits would otherwise plot as a flawless result."""
    return (not b.get('extra', {}).get('error')
            and b.get('qber') is not None
            and b.get('sifted_bits', 0) > 0)

# A sweep archived under the same scenario name can hold entirely different
# physics; §5 draws its expected-QBER line from the CURRENT config, so mixing the
# two silently compares old backends against a new expectation.
_want = {'distance_km': cfg['channel']['distance_km'],
         'attenuation_db_per_km': cfg['channel']['attenuation_db_per_km'],
         'polarization_fidelity': cfg['channel']['polarization_fidelity']}

xval_path = PROJECT_DIR / 'results' / 'all_scenarios.json'
backends = None
if xval_path.exists():
    rows = json.loads(xval_path.read_text())
    match = next((r for r in rows if r.get('scenario') == metrics.scenario_name), None)
    if match is None:
        print(f"all_scenarios.json has no row for '{metrics.scenario_name}' "
              f"({len(rows)} rows) — falling back to a local comparison.")
    else:
        drift = {k: (match.get(k), v) for k, v in _want.items()
                 if match.get(k) is None or abs(float(match[k]) - float(v)) > 1e-9}
        if drift:
            print(f"all_scenarios.json has a row named '{metrics.scenario_name}', but it "
                  f"records different physics — not using it:")
            for k, (got, want) in drift.items():
                print(f"    {k}: sweep={got} vs this run={want}")
            print('  Re-run notebook fabric/04 for this configuration.')
        else:
            usable = [b for b in match['backends'] if _usable(b)]
            rejected = [b for b in match['backends'] if not _usable(b)]
            if usable:
                backends = usable
                print(f"Loaded {len(backends)} backend(s) for '{metrics.scenario_name}' "
                      f"from notebook fabric/04 (all_scenarios.json).")
                for b in rejected:
                    why = b.get('extra', {}).get('error') or 'no sifted bits'
                    print(f"    skipped {b.get('platform')}: {why}")
            else:
                print(f"'{metrics.scenario_name}' row has no usable backend "
                      f"({len(match['backends'])} present) — falling back to a local comparison.")
else:
    print('No all_scenarios.json — run notebook fabric/04 for the on-slice SeQUeNCe/NetSquid comparison.')
    print('Falling back to FABRIC measured vs local QFabric-sim.')

if backends is None:
    sim_scenario = ValidationScenario(
        name=metrics.scenario_name,
        distance_km=cfg['channel']['distance_km'],
        attenuation_db_per_km=cfg['channel']['attenuation_db_per_km'],
        detector_efficiency=cfg['detector']['efficiency'],
        dark_count_rate_hz=cfg['detector']['dark_count_rate'],
        polarization_fidelity=cfg['channel']['polarization_fidelity'],
        num_photons=metrics.photons_sent,
        sample_fraction=cfg['protocol']['sample_fraction'],
        seed=cfg.get('seed', 42),
    )
    sim = run_qfabric_bb84_simulated(sim_scenario)
    # _usable applies here too: this run itself can have sifted nothing, and a
    # qber of 0.0 off zero sifted bits would plot as a flawless 0%.
    backends = [b for b in (
        {'platform': 'qfabric', 'qber': metrics.qber, 'sifted_bits': metrics.sifted_bits,
         'secure_key_rate': metrics.secure_key_rate},
        {'platform': 'qfabric_sim', 'qber': sim.qber, 'sifted_bits': sim.sifted_bits,
         'secure_key_rate': sim.secure_key_rate},
    ) if _usable(b)]
    if not backends:
        print('  the measured run sifted no bits either — nothing to compare.')

# Order the platforms consistently for the plots/table.
_order = ['qfabric', 'qfabric_sim', 'sequence', 'netsquid']
present = ([b for p in _order for b in backends if b['platform'] == p]
           + [b for b in backends if b['platform'] not in _order])
for b in present:
    print(f"  {b['platform']:12s} QBER={b['qber']:.4f}  sifted={b.get('sifted_bits')}  "
          f"SKR={b.get('secure_key_rate', 0):.4f}")
if not present:
    print('\nNo backend produced usable data — section 5 has nothing to plot.')
    print('Re-run notebook fabric/02 (and fabric/04 for the simulator comparison);')
    print('a run that sifts zero bits means the channel or the detector dropped everything.')

In [ ]:
if not present:
    print('(no usable backends — skipping the comparison plots; see the message above)')
else:
    names = [b['platform'] for b in present]
    qbers = [b['qber'] * 100 for b in present]
    skrs  = [b.get('secure_key_rate', 0) for b in present]
    palette = ['steelblue', 'darkorange', 'seagreen', 'crimson', 'slategray']
    colors = [palette[i % len(palette)] for i in range(len(names))]

    fid = cfg['channel']['polarization_fidelity']
    exp_qber = (1 - fid) / 2 * 100   # expected intrinsic QBER, percent

    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
    ax = axes[0]
    ax.bar(names, qbers, color=colors)
    ax.axhline(exp_qber, ls='--', color='gray', label=f'expected (1-F)/2 = {exp_qber:.2f}%')
    for i, q in enumerate(qbers):
        ax.text(i, q, f'{q:.2f}%', ha='center', va='bottom', fontsize=9)
    ax.set_ylabel('QBER (%)'); ax.set_title('QBER by platform'); ax.legend()

    ax = axes[1]
    ax.bar(names, skrs, color=colors)
    for i, s in enumerate(skrs):
        ax.text(i, s, f'{s:.3f}', ha='center', va='bottom', fontsize=9)
    ax.set_ylabel('Secure key rate (bits/photon)'); ax.set_title('Secure key rate by platform')

    for ax in axes:
        ax.grid(axis='y', alpha=0.3); ax.tick_params(axis='x', rotation=15)
    plt.tight_layout()

In [ ]:
cmp_df = pd.DataFrame([
    {'platform': b['platform'], 'QBER': round(b['qber'], 4),
     'sifted_bits': b.get('sifted_bits'),
     'secure_key_rate': round(b.get('secure_key_rate', 0), 4)}
    for b in present
])
cmp_df.style.hide(axis='index')

## 6. Loss Breakdown

Where were photons lost? Breakdown into channel loss, detector
inefficiency, basis mismatch, and QBER sampling overhead.

In [ ]:
# Decompose the photon budget
n_sent = metrics.photons_sent

# Channel survivors (before detector)
channel_survivors = metrics.photons_received / det_eff
lost_channel = n_sent - channel_survivors

# Lost at detector
lost_detector = channel_survivors - metrics.photons_received

# Lost to basis mismatch (detected but not sifted)
lost_basis = metrics.photons_received - metrics.sifted_bits

# Lost to QBER sampling (sifted bits used for estimation, not in final key)
lost_sampling = metrics.sifted_bits - metrics.final_key_bits

# Final key
final_key = metrics.final_key_bits

slices = [lost_channel, lost_detector, lost_basis, lost_sampling, final_key]
labels = [
    f"Channel loss\n({lost_channel:,.0f})",
    f"Detector inefficiency\n({lost_detector:,.0f})",
    f"Basis mismatch\n({lost_basis:,.0f})",
    f"QBER sampling\n({lost_sampling:,.0f})",
    f"Final key\n({final_key:,.0f})",
]
colors = ["#e74c3c", "#e67e22", "#f1c40f", "#95a5a6", "#2ecc71"]

fig, ax = plt.subplots(figsize=(8, 6))
wedges, texts, autotexts = ax.pie(
    slices, labels=labels, colors=colors, autopct="%1.1f%%",
    startangle=140, pctdistance=0.8,
)
for t in autotexts:
    t.set_fontsize(9)
ax.set_title(f"Photon Budget Breakdown (n={n_sent:,} sent)")
plt.tight_layout()